**Legacy / unsupported.** The documented path is **Google Colab T4** only—see **IMPORT_OLLAMA.md** and **colab_full_import.ipynb** in the Project Payphone repository. This Kaggle notebook is kept as an optional escape hatch; behavior may drift from current training exports.

---

# Payphone Full Import – Merge + GGUF on Kaggle (Free GPU)

Merge your LoRA adapter and convert to GGUF on Kaggle's free GPU. Uses **transformers + llama.cpp** (no Unsloth) to avoid install issues.

**Setup:**
1. Create a Kaggle account at [kaggle.com](https://kaggle.com)
2. **Settings** → turn **Internet** ON (required for pip)
3. **Settings** → Accelerator → **GPU P100** or **T4**
4. Create a **Dataset**: upload `payphone-storyteller-lora.zip`, name it `payphone-lora`
5. Add your dataset to this notebook, then run all cells

## 1. Install dependencies

In [ ]:
# Reduce CUDA fragmentation (run first; on Kaggle: Session → Restart if you hit GPU OOM)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# CRITICAL: Upgrade numpy to 2.x first (fixes "numpy.dtype size changed, Expected 96 got 88")
!pip install -q --upgrade "numpy>=2.0"

# Pin torch; keep numpy>=2 so it doesn't get downgraded
import torch
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\nnumpy>=2.0\n")
print(f"torch {torch.__version__} (CUDA: {torch.cuda.is_available()}), numpy {__import__('numpy').__version__}")

# Install our packages; constraints prevent torch changes
# Align with Colab import notebook; use 4.47.1 if 4.47.2 missing on PyPI
!pip install -q -c /tmp/constraints.txt peft bitsandbytes "transformers==4.47.1" "accelerate==1.1.1"
# (Any "dependency conflict" lines about torchaudio/fastai are Kaggle's other packages - safe to ignore)

# Clone llama.cpp for GGUF conversion (skip if already exists from re-run)
# NOTE: Do NOT install llama.cpp/requirements.txt - it pulls torch~=2.6.0 from CPU index
# and overwrites Kaggle's CUDA torch. convert_hf_to_gguf.py uses bundled gguf-py from the repo.
if not os.path.exists("llama.cpp/convert_hf_to_gguf.py"):
    !git clone -q https://github.com/ggerganov/llama.cpp

## 2. Locate LoRA adapter from dataset

In [ ]:
import os
import zipfile

INPUT_ROOT = "/kaggle/input"
WORKING = "/kaggle/working"

def has_adapter(p):
    return os.path.exists(os.path.join(p, "adapter_config.json")) and os.path.exists(os.path.join(p, "adapter_model.safetensors"))

# First: recursive search for adapter_config.json anywhere under /kaggle/input
adapter_path = None
for root, dirs, files in os.walk(INPUT_ROOT):
    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
        adapter_path = root
        print(f"Found adapter at: {adapter_path}")
        break

# If not found, try extracting zips
if not adapter_path:
    print("Scanning datasets...")
    for name in sorted(os.listdir(INPUT_ROOT)):
        d = os.path.join(INPUT_ROOT, name)
        if not os.path.isdir(d):
            continue
        contents = os.listdir(d)
        print(f"  {name}: {contents}")
        for f in contents:
            if f.endswith(".zip"):
                zip_path = os.path.join(d, f)
                extract_dir = os.path.join(WORKING, "lora")
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(extract_dir)
                for root, _, files in os.walk(extract_dir):
                    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
                        adapter_path = root
                        break
                if adapter_path:
                    break
        if adapter_path:
            break

if not adapter_path or not has_adapter(adapter_path):
    print("\nYour dataset must contain adapter_config.json and adapter_model.safetensors")
    print("Create new dataset: New Dataset -> Upload -> select payphone-storyteller-lora folder or zip")
    print("Then Add Data to this notebook and re-run.")
    raise FileNotFoundError("LoRA adapter not found.")
print(f"Adapter at: {adapter_path}")

## 3. Merge LoRA and convert to GGUF

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModelForCausalLM
import torch

BASE = "Qwen/Qwen2.5-7B-Instruct"
MERGED_DIR = os.path.join(WORKING, "merged_payphone")
OFFLOAD_DIR = os.path.join(WORKING, "offload")
GGUF_PATH = os.path.join(WORKING, "payphone-story.gguf")
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
print("Loading base model (4-bit, GPU+CPU offload)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "14GiB", "cpu": "60GiB"},
    offload_folder=OFFLOAD_DIR,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

print("Loading LoRA adapter...")
model = PeftModelForCausalLM.from_pretrained(model, adapter_path)

print("Merging...")
model = model.merge_and_unload()

print("Saving merged model (HF format)...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
del model
gc.collect()
torch.cuda.empty_cache()

print("Converting to GGUF (q8_0) via llama.cpp...")
import subprocess
subprocess.run(
    ["python", "llama.cpp/convert_hf_to_gguf.py", MERGED_DIR, "--outfile", GGUF_PATH, "--outtype", "q8_0"],
    check=True, cwd="/kaggle/working"
)
print("Done.")

## 4. Download GGUF

In [ ]:
from IPython.display import FileLink, display

gguf_path = os.path.join(WORKING, "payphone-story.gguf")
if os.path.exists(gguf_path):
    os.chdir(WORKING)
    print("GGUF ready. Click the link below to download:")
    display(FileLink("payphone-story.gguf", result_html_prefix="Download: "))
    print("Place in project root, then: ollama create payphone-story -f Modelfile")
else:
    print("ERROR: GGUF not found. Check the merge cell above for errors.")